# Matemáticas de la Inteligencia Artificial
## Sesión 14 — Entrenar un mini-GPT: predicción del siguiente token de extremo a extremo

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CuentosCuanticos/matematicas-ia/blob/main/14_entrenamiento_mini_gpt/laboratorio.ipynb)

**Pregunta de la sesión:** ¿cómo convertimos un Transformer ya construido en un modelo de lenguaje que aprende de un corpus?

Cerraremos el circuito completo: corpus → tokens → ventanas → mini-GPT → logits → cross-entropy → backpropagation → AdamW → validación.

El objetivo es que puedas explicar qué representa cada tensor y por qué cada operación es necesaria. No usaremos `nn.Transformer`: las piezas fundamentales quedan visibles.

In [ ]:
import math, random, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 14
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| dispositivo:', device)

## 1. Del texto a una secuencia de índices

Trabajaremos con un corpus sintético y transparente para no depender de descargas. La tokenización será por caracteres. Los índices son etiquetas discretas; la geometría aparecerá después en los embeddings.

In [ ]:
subjects = ['el gradiente','la atencion','el embedding','la probabilidad','el transformer','la matriz','el contexto','la perdida','el optimizador','la validacion','el token','la capa residual']
verbs = ['transforma','compara','propaga','resume','pondera','aprende','organiza','aproxima','conecta','normaliza']
objects = ['informacion del contexto','vectores de representacion','errores de prediccion','dependencias entre posiciones','una distribucion de probabilidad','parametros del modelo','senales de entrenamiento','patrones del corpus']
ends = ['durante el entrenamiento.','antes de la siguiente actualizacion.','sin mirar los tokens futuros.','y despues medimos en validacion.','para predecir el siguiente simbolo.']

sentences=[]
for i in range(720):
    sentences.append(f'{subjects[i%len(subjects)]} {verbs[(3*i+1)%len(verbs)]} {objects[(5*i+2)%len(objects)]} {ends[(7*i+3)%len(ends)]}')
rng=random.Random(SEED); rng.shuffle(sentences)
text='\n'.join(sentences)
chars=sorted(set(text)); V=len(chars)
stoi={c:i for i,c in enumerate(chars)}; itos={i:c for c,i in stoi.items()}
encode=lambda s:[stoi[c] for c in s]
decode=lambda ids:''.join(itos[int(i)] for i in ids)
data=torch.tensor(encode(text),dtype=torch.long)
print('tokens:',len(data),'| V:',V,'| muestra:',repr(text[:140]))

## 2. Train/validation antes de formar ventanas

Primero dividimos la secuencia y después extraemos ventanas. Así evitamos que ventanas casi idénticas terminen artificialmente una en train y otra en validation. Para longitud de contexto T construimos X=(t_s,…,t_{s+T-1}) e Y=(t_{s+1},…,t_{s+T}).

In [ ]:
n=int(0.90*len(data)); train_data=data[:n]; val_data=data[n:]
BATCH_SIZE=32; BLOCK_SIZE=64

def get_batch(split,batch_size=BATCH_SIZE,block_size=BLOCK_SIZE):
    source=train_data if split=='train' else val_data
    starts=torch.randint(0,len(source)-block_size-1,(batch_size,))
    x=torch.stack([source[i:i+block_size] for i in starts])
    y=torch.stack([source[i+1:i+block_size+1] for i in starts])
    return x.to(device),y.to(device)

x0,y0=get_batch('train',batch_size=1,block_size=20)
print('X:',repr(decode(x0[0].cpu())))
print('Y:',repr(decode(y0[0].cpu())))
assert torch.equal(x0[0,1:].cpu(),y0[0,:-1].cpu())

### Tarea guiada 1
Comprueba con una ventana de longitud 12 que y[i]=x[i+1] para i=0,…,10. Escribe la comprobación en la celda siguiente; no es necesaria para ejecutar el resto del cuaderno.

In [ ]:
# TU COMPROBACIÓN AQUÍ
print('Tarea guiada 1 pendiente.')

## 3. Mini-GPT decoder-only

La atención causal usa A=softmax(QK^T/sqrt(d_h)+M). Cada bloque aplica atención y una MLP con conexiones residuales. La cabeza final proyecta de dimensión d_model a V logits.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self,d_model,n_heads,block_size,dropout=0.0):
        super().__init__(); assert d_model%n_heads==0
        self.n_heads=n_heads; self.head_dim=d_model//n_heads
        self.qkv=nn.Linear(d_model,3*d_model,bias=False)
        self.proj=nn.Linear(d_model,d_model,bias=False)
        self.dropout=nn.Dropout(dropout)
        mask=torch.tril(torch.ones(block_size,block_size,dtype=torch.bool))
        self.register_buffer('mask',mask.view(1,1,block_size,block_size))
    def forward(self,x):
        B,T,C=x.shape
        q,k,v=self.qkv(x).chunk(3,dim=-1)
        q=q.view(B,T,self.n_heads,self.head_dim).transpose(1,2)
        k=k.view(B,T,self.n_heads,self.head_dim).transpose(1,2)
        v=v.view(B,T,self.n_heads,self.head_dim).transpose(1,2)
        scores=(q@k.transpose(-2,-1))/math.sqrt(self.head_dim)
        scores=scores.masked_fill(~self.mask[:,:,:T,:T],float('-inf'))
        A=self.dropout(F.softmax(scores,dim=-1))
        out=A@v
        out=out.transpose(1,2).contiguous().view(B,T,C)
        return self.proj(out)

class FeedForward(nn.Module):
    def __init__(self,d_model,dropout=0.0):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(d_model,4*d_model),nn.GELU(),nn.Linear(4*d_model,d_model),nn.Dropout(dropout))
    def forward(self,x): return self.net(x)

class Block(nn.Module):
    def __init__(self,d_model,n_heads,block_size,dropout=0.0):
        super().__init__(); self.ln1=nn.LayerNorm(d_model); self.ln2=nn.LayerNorm(d_model)
        self.attn=CausalSelfAttention(d_model,n_heads,block_size,dropout); self.ff=FeedForward(d_model,dropout)
    def forward(self,x):
        x=x+self.attn(self.ln1(x)); x=x+self.ff(self.ln2(x)); return x

class MiniGPT(nn.Module):
    def __init__(self,vocab_size,block_size=64,d_model=64,n_heads=4,n_layers=2,dropout=0.05):
        super().__init__(); self.block_size=block_size
        self.tok_emb=nn.Embedding(vocab_size,d_model); self.pos_emb=nn.Embedding(block_size,d_model)
        self.blocks=nn.Sequential(*[Block(d_model,n_heads,block_size,dropout) for _ in range(n_layers)])
        self.ln_f=nn.LayerNorm(d_model); self.lm_head=nn.Linear(d_model,vocab_size)
    def forward(self,idx):
        B,T=idx.shape; pos=torch.arange(T,device=idx.device)
        x=self.tok_emb(idx)+self.pos_emb(pos); x=self.blocks(x); x=self.ln_f(x)
        return self.lm_head(x)
    @torch.no_grad()
    def generate(self,idx,max_new_tokens=100,temperature=1.0):
        for _ in range(max_new_tokens):
            logits=self(idx[:,-self.block_size:])[:,-1,:]/max(temperature,1e-6)
            nxt=torch.multinomial(F.softmax(logits,dim=-1),1); idx=torch.cat([idx,nxt],dim=1)
        return idx

## 4. De logits a cross-entropy

El recorrido de dimensiones es B×T → B×T×d_model → B×T×V → 1. Para una distribución inicialmente casi uniforme esperamos una pérdida del orden de log(V).

In [ ]:
model=MiniGPT(V,block_size=BLOCK_SIZE,d_model=64,n_heads=4,n_layers=2).to(device)
x,y=get_batch('train'); logits=model(x)
loss=F.cross_entropy(logits.reshape(-1,V),y.reshape(-1))
print('X:',x.shape,'| logits:',logits.shape,'| Y:',y.shape)
print('loss inicial:',float(loss),'| log(V):',math.log(V))

### Tarea guiada 2
Elige una posición, calcula softmax manualmente, toma la probabilidad del token correcto y verifica que -log(p_correcto) coincide con `F.cross_entropy` para esa posición.

In [ ]:
# TU COMPROBACIÓN AQUÍ
print('Tarea guiada 2 pendiente.')

## 5. Backpropagation y AdamW

`backward()` calcula gradientes; `step()` modifica parámetros. Verificamos que el error llega a embeddings, atención, MLP y LM head.

In [ ]:
model.zero_grad(set_to_none=True)
logits=model(x); loss=F.cross_entropy(logits.reshape(-1,V),y.reshape(-1)); loss.backward()
checks={'embedding':model.tok_emb.weight.grad.norm().item(),'qkv':model.blocks[0].attn.qkv.weight.grad.norm().item(),'MLP':model.blocks[0].ff.net[0].weight.grad.norm().item(),'LM head':model.lm_head.weight.grad.norm().item()}
for k,v in checks.items(): print(f'{k:10s}: {v:.6f}')
assert all(v>0 for v in checks.values())

## 6. Evaluar sin entrenar y guardar el mejor estado

Validation se mide con parámetros fijos. El mejor checkpoint se elige por mínima `val loss`, no por ser el último.

In [ ]:
@torch.no_grad()
def estimate_loss(model,eval_iters=10,batch_size=32,block_size=64):
    out={}; model.eval()
    for split in ('train','val'):
        vals=[]
        for _ in range(eval_iters):
            xb,yb=get_batch(split,batch_size,block_size); z=model(xb)
            vals.append(F.cross_entropy(z.reshape(-1,V),yb.reshape(-1)).item())
        out[split]=float(np.mean(vals))
    model.train(); return out

def train_model(config,steps=180,eval_interval=30,seed=SEED,verbose=True):
    torch.manual_seed(seed)
    m=MiniGPT(V,config['block_size'],config['d_model'],config['n_heads'],config['n_layers'],config.get('dropout',0.05)).to(device)
    opt=torch.optim.AdamW(m.parameters(),lr=config.get('lr',2e-3),weight_decay=config.get('weight_decay',1e-2))
    hist={'step':[],'train':[],'val':[]}; best=float('inf'); best_state=None
    for step in range(steps+1):
        if step%eval_interval==0:
            e=estimate_loss(m,8,config.get('batch_size',32),config['block_size'])
            hist['step'].append(step); hist['train'].append(e['train']); hist['val'].append(e['val'])
            if verbose: print(f"step {step:4d} | train {e['train']:.3f} | val {e['val']:.3f}")
            if e['val']<best: best=e['val']; best_state=copy.deepcopy(m.state_dict())
        if step==steps: break
        xb,yb=get_batch('train',config.get('batch_size',32),config['block_size']); z=m(xb)
        L=F.cross_entropy(z.reshape(-1,V),yb.reshape(-1)); opt.zero_grad(set_to_none=True); L.backward(); opt.step()
    m.load_state_dict(best_state); return m,hist,best

In [ ]:
config={'block_size':64,'d_model':64,'n_heads':4,'n_layers':2,'dropout':0.05,'batch_size':32,'lr':2e-3,'weight_decay':1e-2}
model_trained,hist,best_val=train_model(config)
plt.figure(figsize=(7,4)); plt.plot(hist['step'],hist['train'],marker='o',label='train'); plt.plot(hist['step'],hist['val'],marker='o',label='validation'); plt.xlabel('paso'); plt.ylabel('cross-entropy'); plt.legend(); plt.grid(alpha=.25); plt.show()
print('mejor val loss:',best_val,'| gap final:',hist['val'][-1]-hist['train'][-1])

### Tarea guiada 3 — Diagnóstico
Explica: (a) si hay underfitting al comienzo; (b) si train y validation se separan; (c) por qué el mejor checkpoint no tiene por qué ser el último.

## 7. Parámetros frente a coste de atención

Con MLP de anchura 4d, un bloque tiene aproximadamente 12 d² parámetros. En cambio, QK^T tiene forma T×T: la self-attention densa contiene un término O(T²d). Duplicar T cuadruplica el número de compatibilidades.

In [ ]:
def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
print('modelo base:',f'{count_params(model_trained):,}','parámetros')
for T in [32,64,128,256]:
    m=MiniGPT(V,block_size=T,d_model=64,n_heads=4,n_layers=2)
    print(f'T={T:3d} | params={count_params(m):,} | T²={T*T:,}')
Ts=np.array([16,32,64,128,256,512]); plt.figure(figsize=(6,4)); plt.plot(Ts,Ts**2,marker='o'); plt.xlabel('T'); plt.ylabel('T²'); plt.grid(alpha=.25); plt.show()

## 8. Primera muestra
La generación se estudiará en profundidad en la sesión 15; aquí solo comprobamos que el checkpoint ya define una distribución autorregresiva utilizable.

In [ ]:
prompt='el transformer '; start=torch.tensor([encode(prompt)],dtype=torch.long,device=device)
out=model_trained.generate(start,140,0.9); print(decode(out[0].cpu()))

# Problema final abierto — Diseñar un mini-GPT bajo presupuesto

Dispones del corpus del cuaderno y de un presupuesto máximo de **200 000 parámetros entrenables**. Obtén la menor `validation loss` posible.

**Requisitos:** compara al menos dos arquitecturas; ninguna puede superar 200 000 parámetros; usa como máximo 140 actualizaciones por candidata; registra train/validation; selecciona por mejor validación; guarda `mejor_modelo_s14.pt`; genera desde `la atencion `; entrega tabla, gráfica y una conclusión de 8–12 líneas justificando arquitectura, contexto, generalización y un experimento siguiente.

No existe una solución única: se evalúa la calidad del diseño experimental y de la interpretación.

In [ ]:
# ESPACIO DE TRABAJO DEL PROBLEMA FINAL
final_configs={
    # 'modelo_A': {...},
    # 'modelo_B': {...},
}
print('Problema final pendiente: define tus configuraciones y usa train_model(...).')

## Cierre
El entrenamiento completo es: datos de train → pérdida → gradientes → AdamW → nuevos parámetros; los datos de validación → pérdida de validación → diagnóstico, sin actualización. La sesión 15 partirá del checkpoint y estudiará el muestreo autorregresivo.